# Computer Exercise 15.27 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.27 Sequential Decision Making — *Component-Diagnosis by Init-Scale × Learning-Rate Grid*
> **풀이 일자**: Day 94
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 1.** Day 93 (§15.26 Problem 1) reported that under a **single fixed learning rate**
> $\eta = 0.05$, three of the four +CNRT components — Cramér, EMA whitening, and Twin-split —
> produced individually **negative** marginal effects when toggled on top of the plain MSE
> baseline, with EMA whitening in particular collapsing catastrophically (marginal $\approx -2.55$).
> That result assumed all components share the baseline's step size, which may confound
> "component itself is toxic" with "component's dynamics need a different step size."
> **Sweep a two-dimensional grid of initialization scale**
> $\sigma_0 \in \{0.1, 0.3, 0.5\}$ **and learning rate** $\eta \in \{0.01, 0.02, 0.05, 0.10\}$
> and rerun the four single-component ablations (baseline + Noisy, baseline + Cramér,
> baseline + EMA, baseline + Twin) for 600 training steps, 3 seeds each, softmax deployment at
> $\tau=0.10$. Report (a) each component's **best cell**
> $(\sigma_0^\star, \eta^\star) = \arg\max_{\sigma_0, \eta} R^{(c)}$, (b) the **rehabilitation gap**
> $\rho_c = R^{(c)}(\sigma_0^\star, \eta^\star) - R^{(c)}(\sigma_0=0.5, \eta=0.05)$ measuring how
> much Day 93's fixed setup handicapped each component, and (c) whether any component matches
> or exceeds the baseline once given its preferred hyperparameters.

### 한국어 풀이용 정리
Day 93 P1 에서 개별 성분의 marginal effect 가 모두 음수로 나온 것이 **성분 자체의 결함**인지
**성분에 맞지 않는 lr / 초기화 스케일** 때문인지를 구분한다. 3×4 = 12 개의 하이퍼셀에서 각
성분을 재훈련하고, 각 성분의 최적 셀과 Day 93 default 셀 사이의 성능 차 (rehabilitation gap
$\rho_c$) 를 측정.


## 2. 수학적 배경

### 2.1 처방 성분과 목적
$c \in \{\text{Noisy}, \text{Cramér}, \text{EMA}, \text{Twin}\}$. 각 성분은 baseline 위에
단독 부착. Day 93 은 $\sigma_0=0.5,\ \eta=0.05$ 로 고정했음.

### 2.2 격자와 시드
$$
G = \{(\sigma_0, \eta) : \sigma_0 \in \{0.1, 0.3, 0.5\},\ \eta \in \{0.01, 0.02, 0.05, 0.10\}\}
$$
각 셀에서 3 시드 (94101–94103) × 5 (baseline + 4 성분) = 15 실행. 총 12 셀 × 15 = 180 실행.

### 2.3 지표
- 셀별 tail-8 return $R^{(c)}(\sigma_0, \eta)$: 60 에피소드의 마지막 8 개 평균, 3 시드 평균.
- 최적 셀 $(\sigma_0^\star, \eta^\star)_c = \arg\max R^{(c)}$.
- $\rho_c = R^{(c)}(\sigma_0^\star, \eta^\star) - R^{(c)}(0.5, 0.05)$.
- $\Delta_c^\star = R^{(c)}(\sigma_0^\star, \eta^\star) - R^{\text{base}}(\sigma_0^\star, \eta^\star)$.


## 3. 풀이 흐름

1. Chain MDP + softmax 배포 wrapper 재사용 (Day 93 P1 과 동일).
2. `Learner(component, sigma0, eta, seed)` — 단일 성분 부착.
3. 12 셀 격자에서 4 성분 + baseline 각 3 시드 학습 (600 step).
4. 성분별 heatmap ($\sigma_0 \times \eta$) 시각화.
5. 최적 셀과 rehabilitation gap 표.
6. Day 93 default vs 셀-optimal 비교 bar chart.
7. 결론.


In [1]:

import os
os.environ['MPLCONFIGDIR'] = '/tmp/mplcfg'
os.makedirs('/tmp/mplcfg', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

class ChainMDP:
    def __init__(self, N=5, p_slip=0.10, step_r=-0.02, goal_r=1.0, rng=None):
        self.N, self.p_slip, self.step_r, self.goal_r = N, p_slip, step_r, goal_r
        self.rng = rng or np.random.default_rng(0)
    def reset(self):
        self.s = 0; return self.s
    def step(self, a):
        if self.rng.random() < self.p_slip:
            a = 1 - a
        if a == 1: self.s = min(self.s + 1, self.N - 1)
        else:      self.s = max(self.s - 1, 0)
        done = (self.s == self.N - 1)
        r = self.goal_r if done else self.step_r
        return self.s, r, done

def one_hot(s, N):
    x = np.zeros(N); x[s] = 1.0; return x


class Learner:
    def __init__(self, seed=0, H=16, N=5, A=2, K=10,
                 component='none', sigma0=0.5, eta=0.05, gamma=0.95, eps=0.20):
        self.rng = np.random.default_rng(seed)
        self.H, self.N, self.A, self.K = H, N, A, K
        self.comp = component
        self.eta = eta; self.gamma = gamma; self.eps = eps
        self.W1 = self.rng.normal(0, sigma0, size=(N, H)); self.b1 = np.zeros(H)
        out_dim = K if component == 'cramer' else 1
        if component == 'twin':
            self.KA = 5; self.KB = 5
            self.W2A = [self.rng.normal(0, sigma0, size=(H, self.KA)) for _ in range(A)]
            self.b2A = [np.zeros(self.KA) for _ in range(A)]
            self.W2B = [self.rng.normal(0, sigma0, size=(H, self.KB)) for _ in range(A)]
            self.b2B = [np.zeros(self.KB) for _ in range(A)]
            self.atomsA = np.linspace(-1.0, 1.0, 5)
            self.atomsB = np.linspace(-1.0, 1.0, 5)
        else:
            self.W2 = [self.rng.normal(0, sigma0, size=(H, out_dim)) for _ in range(A)]
            self.b2 = [np.zeros(out_dim) for _ in range(A)]
        if component == 'noisy':
            self.sW2 = [np.full((H, out_dim), 0.1) for _ in range(A)]
            self.sb2 = [np.full(out_dim, 0.1) for _ in range(A)]
        if component == 'ema':
            self.mu = np.zeros(H); self.var = np.ones(H); self.beta = 0.99
        if component == 'cramer':
            self.atoms = np.linspace(-1.0, 1.0, K)

    def phi(self, s):
        x = one_hot(s, self.N)
        h = np.tanh(self.W1.T @ x + self.b1)
        if self.comp == 'ema':
            h = (h - self.mu) / np.sqrt(self.var + 1e-6)
        return h

    def q_scalar(self, s):
        h = self.phi(s)
        out = []
        for a in range(self.A):
            if self.comp == 'cramer':
                z = h @ self.W2[a] + self.b2[a]
                p = np.exp(z - z.max()); p/=p.sum()
                q = float((p * self.atoms).sum())
            elif self.comp == 'twin':
                lA = h @ self.W2A[a] + self.b2A[a]; pA = np.exp(lA - lA.max()); pA/=pA.sum()
                lB = h @ self.W2B[a] + self.b2B[a]; pB = np.exp(lB - lB.max()); pB/=pB.sum()
                qA = float((pA * self.atomsA).sum()); qB = float((pB * self.atomsB).sum())
                q = 0.5 * (qA + qB)
            elif self.comp == 'noisy':
                eps_w = self.rng.standard_normal(self.W2[a].shape)
                eps_b = self.rng.standard_normal(self.b2[a].shape)
                W = self.W2[a] + self.sW2[a] * eps_w
                b = self.b2[a] + self.sb2[a] * eps_b
                q = float((h @ W + b)[0])
            else:
                q = float((h @ self.W2[a] + self.b2[a])[0])
            out.append(q)
        return np.array(out)

    def update_ema(self, h_raw):
        self.mu = self.beta * self.mu + (1 - self.beta) * h_raw
        self.var = self.beta * self.var + (1 - self.beta) * (h_raw - self.mu) ** 2

    def train_step(self, s, a, r, sp, done):
        x = one_hot(s, self.N)
        h_raw = np.tanh(self.W1.T @ x + self.b1)
        if self.comp == 'ema':
            self.update_ema(h_raw)
            h = (h_raw - self.mu) / np.sqrt(self.var + 1e-6)
        else:
            h = h_raw
        with_boot = 0.0 if done else float(self.q_scalar(sp).max())
        target = r + self.gamma * with_boot
        if self.comp == 'cramer':
            z = h @ self.W2[a] + self.b2[a]
            m = np.exp(z - z.max()); m/=m.sum()
            t = np.clip(target, self.atoms[0], self.atoms[-1])
            idx = np.searchsorted(self.atoms, t); idx = max(1, min(self.K - 1, idx))
            lo, hi = self.atoms[idx-1], self.atoms[idx]
            w_hi = (t - lo) / (hi - lo + 1e-12)
            tgt = np.zeros(self.K); tgt[idx-1] = 1 - w_hi; tgt[idx] = w_hi
            grad_z = m - tgt
            self.W2[a] -= self.eta * np.outer(h, grad_z); self.b2[a] -= self.eta * grad_z
            grad_h = self.W2[a] @ grad_z
        elif self.comp == 'twin':
            gW1_agg = np.zeros_like(self.W1); gb1_agg = np.zeros_like(self.b1)
            for atoms_ref, W2_ref, b2_ref in [(self.atomsA, self.W2A, self.b2A), (self.atomsB, self.W2B, self.b2B)]:
                z = h @ W2_ref[a] + b2_ref[a]
                m = np.exp(z - z.max()); m/=m.sum()
                Kr = len(atoms_ref)
                t = np.clip(target, atoms_ref[0], atoms_ref[-1])
                idx = np.searchsorted(atoms_ref, t); idx = max(1, min(Kr - 1, idx))
                lo, hi = atoms_ref[idx-1], atoms_ref[idx]
                w_hi = (t - lo) / (hi - lo + 1e-12)
                tgt = np.zeros(Kr); tgt[idx-1] = 1 - w_hi; tgt[idx] = w_hi
                grad_z = m - tgt
                W2_ref[a] -= self.eta * np.outer(h, grad_z); b2_ref[a] -= self.eta * grad_z
                grad_h_part = W2_ref[a] @ grad_z
                d = grad_h_part * (1 - h_raw**2)
                gW1_agg += np.outer(x, d); gb1_agg += d
            self.W1 -= self.eta * gW1_agg; self.b1 -= self.eta * gb1_agg
            return
        elif self.comp == 'noisy':
            eps_w = self.rng.standard_normal(self.W2[a].shape)
            eps_b = self.rng.standard_normal(self.b2[a].shape)
            W = self.W2[a] + self.sW2[a] * eps_w
            b = self.b2[a] + self.sb2[a] * eps_b
            q_now = float((h @ W + b)[0])
            err = q_now - target
            grad_out = np.array([err])
            self.W2[a] -= self.eta * np.outer(h, grad_out); self.b2[a] -= self.eta * grad_out
            self.sW2[a] -= self.eta * np.outer(h, grad_out) * eps_w
            self.sb2[a] -= self.eta * grad_out * eps_b
            grad_h = W @ grad_out
        else:  # baseline / ema
            q_now = float((h @ self.W2[a] + self.b2[a])[0])
            err = q_now - target
            grad_out = np.array([err])
            self.W2[a] -= self.eta * np.outer(h, grad_out); self.b2[a] -= self.eta * grad_out
            grad_h = self.W2[a] @ grad_out
        d = grad_h * (1 - h_raw**2)
        self.W1 -= self.eta * np.outer(x, d); self.b1 -= self.eta * d

    def act(self, s, tau=None, eps=None):
        q = self.q_scalar(s)
        if tau is not None and tau > 0:
            p = np.exp(q/tau - (q/tau).max()); p /= p.sum()
            return int(self.rng.choice(self.A, p=p))
        if eps is None: eps = self.eps
        if self.rng.random() < eps: return int(self.rng.integers(self.A))
        return int(np.argmax(q))


def train_and_eval(seed, component, sigma0, eta, T=600, tau=0.10, n_eval=60):
    env = ChainMDP(rng=np.random.default_rng(seed + 7))
    L = Learner(seed=seed, component=component, sigma0=sigma0, eta=eta)
    s = env.reset()
    for _ in range(T):
        a = L.act(s, eps=0.20)
        sp, r, done = env.step(a)
        L.train_step(s, a, r, sp, done)
        s = env.reset() if done else sp
    rets = []
    for ep in range(n_eval):
        env.rng = np.random.default_rng(seed + 1000 + ep)
        s = env.reset(); total = 0.0
        for _ in range(50):
            a = L.act(s, tau=tau)
            s, r, done = env.step(a)
            total += r
            if done: break
        rets.append(total)
    return float(np.mean(rets[-8:]))
print("Learner ready.")


/tmp/mplcfg is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-vdjagxts because there was an issue with the default path (/tmp/mplcfg); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Learner ready.


In [2]:

SEEDS = [94101, 94102, 94103]
COMPS = ['baseline', 'noisy', 'cramer', 'ema', 'twin']
SIGMAS = [0.1, 0.3, 0.5]
ETAS = [0.01, 0.02, 0.05, 0.10]
results = []
for comp in COMPS:
    cname = 'none' if comp == 'baseline' else comp
    for sig in SIGMAS:
        for eta in ETAS:
            rs = [train_and_eval(sd, cname, sig, eta, T=600) for sd in SEEDS]
            results.append({'component': comp, 'sigma0': sig, 'eta': eta,
                            'R_mean': float(np.mean(rs)), 'R_std': float(np.std(rs))})
df = pd.DataFrame(results)
df


,component,sigma0,eta,R_mean,R_std
0,baseline,0.1000,0.0100,0.5992,0.4431
1,baseline,0.1000,0.0200,0.6525,0.3677
2,baseline,0.1000,0.0500,0.8775,0.0122
3,baseline,0.1000,0.1000,0.8575,0.0334
4,baseline,0.3000,0.0100,0.1575,0.7163
5,baseline,0.3000,0.0200,0.5408,0.5029
6,baseline,0.3000,0.0500,0.8658,0.0130
7,baseline,0.3000,0.1000,0.8617,0.0295
8,baseline,0.5000,0.0100,0.3850,0.6948
9,baseline,0.5000,0.0200,0.8358,0.0051


In [3]:

DEFAULT_SIG, DEFAULT_ETA = 0.5, 0.05
summary = []
for comp in COMPS:
    sub = df[df.component == comp]
    best_row = sub.loc[sub.R_mean.idxmax()]
    default_row = sub[(sub.sigma0==DEFAULT_SIG) & (sub.eta==DEFAULT_ETA)].iloc[0]
    base_at_best = df[(df.component=='baseline') & (df.sigma0==best_row.sigma0) & (df.eta==best_row.eta)].R_mean.values[0]
    summary.append({
        'component': comp,
        'best_sigma0': best_row.sigma0, 'best_eta': best_row.eta,
        'R_best': best_row.R_mean, 'R_default': default_row.R_mean,
        'rehab_gap': best_row.R_mean - default_row.R_mean,
        'delta_vs_base_at_best': best_row.R_mean - base_at_best,
    })
sdf = pd.DataFrame(summary); sdf


,component,best_sigma0,best_eta,R_best,R_default,rehab_gap,delta_vs_base_at_best
0,baseline,0.5000,0.1000,0.8992,0.8675,0.0317,0.0000
1,noisy,0.1000,0.0200,0.9050,0.5750,0.3300,0.2525
2,cramer,0.3000,0.0500,0.9125,0.9033,0.0092,0.0467
3,ema,0.3000,0.0200,0.9075,-0.0442,0.9517,0.3667
4,twin,0.1000,0.0500,0.9142,0.7858,0.1283,0.0367


In [4]:

fig, axes = plt.subplots(1, 5, figsize=(20, 3.6), constrained_layout=True)
vmin, vmax = df.R_mean.min(), df.R_mean.max()
for ax, comp in zip(axes, COMPS):
    sub = df[df.component == comp].pivot(index='sigma0', columns='eta', values='R_mean')
    im = ax.imshow(sub.values, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(ETAS))); ax.set_xticklabels(ETAS)
    ax.set_yticks(range(len(SIGMAS))); ax.set_yticklabels(SIGMAS)
    ax.set_xlabel('learning rate (eta)')
    ax.set_ylabel('init scale (sigma0)')
    ax.set_title(f'component: {comp}')
    for i, sig in enumerate(SIGMAS):
        for j, e in enumerate(ETAS):
            v = sub.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha='center', va='center',
                    color='white' if v < (vmin+vmax)/2 else 'black', fontsize=8)
fig.colorbar(im, ax=axes.tolist(), shrink=0.8, label='tail-8 R')
fig.suptitle('Day 94 P1 — Component rehab: R over (sigma0, eta) grid', y=1.06)
plt.savefig('/tmp/day94_p1_heat.png', dpi=90, bbox_inches='tight'); plt.show()


In [5]:

base_default = df[(df.component=='baseline') & (df.sigma0==DEFAULT_SIG) & (df.eta==DEFAULT_ETA)].R_mean.values[0]
x = np.arange(len(COMPS)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, sdf.R_default.values, width=w, label='Day 93 default cell', color='#888')
ax.bar(x + w/2, sdf.R_best.values, width=w, label='component-optimal cell', color='#c14')
ax.axhline(base_default, ls='--', color='k', lw=0.8, label='baseline at default cell')
ax.set_xticks(x); ax.set_xticklabels(COMPS)
ax.set_ylabel('tail-8 return (softmax tau=0.10)')
ax.set_title('Day 94 P1 — Rehabilitation gap per component')
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig('/tmp/day94_p1_bar.png', dpi=90, bbox_inches='tight'); plt.show()


## 4. 결과 해석

1. **성분별 최적 셀은 default 셀과 다르다** — heatmap 을 보면 어떤 성분도 $(\sigma_0=0.5,\eta=0.05)$
   가 최적 셀이 아니다. 특히 EMA 는 훨씬 작은 $\eta$ 를 선호하는 경향이 뚜렷.
2. **Rehabilitation gap $\rho_c$** — 성분별로 default 대비 최적 셀에서 얼마만큼의 성능을
   회복하는지가 다르다. 큰 $\rho_c$ 는 Day 93 default 가 그 성분을 심하게 handicap 했음을 의미.
3. **$\Delta_c^\star$** (동일 셀 baseline 대비) — 셀-optimal 후에도 baseline 을 넘어서는지는
   성분마다 다르며, 이는 성분 자체의 순수 기여와 하이퍼셀 sensitivity 를 분리한 결과.

> **결론**: Day 93 의 개별 성분 독성은 상당 부분 **셀 mismatch** 에 기인. 셀별 최적 하이퍼를
> 선택하면 대부분의 성분이 회복되며, 이는 조합 (+CNRT) 이 개별 성분들의 단순 합이 아님을 다시
> 확인시킨다.

다음 문제 (P2) 에서는 Day 93 P2 의 Cramér 우위가 **non-tabular (neural) head** 에서도 유지되는지
검증한다.
